# 🩺 AI Skin Disease Detection System — Training Notebook

This notebook trains an **EfficientNetV2B0** model on the **HAM10000** dataset for skin lesion classification.  
It mirrors **`train.py`** so both files stay in sync.

**Results are saved to:** `data/results/`

---

## 1. Mount Google Drive & Setup Environment

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Set working directory to src/ (where train.py, model.py, data_loader.py live)
PROJECT_ROOT = '/content/drive/MyDrive/AI Skin Disease Detection System'
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
os.chdir(SRC_DIR)
print(f'Working directory: {os.getcwd()}')
print(f'Contents: {os.listdir(".")}')

In [ ]:
# Install dependencies
!pip install -q tensorflow pandas scikit-learn matplotlib seaborn opencv-python pillow

## 2. Imports & Configuration

Same imports as `train.py`.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Local imports (same directory — matches train.py)
from data_loader import SkinLesionDataLoader
from model import build_efficientnetv2_model, compile_model

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ──────────────────────────────────────────────
#  Paths  (relative to src/, same defaults as train.py argparse)
# ──────────────────────────────────────────────
DATA_DIR        = '../data/images'
CSV_PATH        = '../data/HAM10000_metadata.csv'
RESULTS_DIR     = '../data/results'          # ← data/results/, NOT src/results/
MODEL_SAVE_PATH = '../models/best_model.h5'
LOG_DIR         = './logs'

# Training hyper-parameters (same defaults as train.py)
EPOCHS     = 20
BATCH_SIZE = 32

# ──────────────────────────────────────────────
#  Image Quality & Font Settings
#  (font: min 10, max 12  —  matches train.py globals)
# ──────────────────────────────────────────────
DPI           = 300
FONT_SIZE_MIN = 10
FONT_SIZE_MAX = 12

matplotlib.rcParams.update({
    'font.size':          FONT_SIZE_MIN,
    'axes.titlesize':     FONT_SIZE_MAX,
    'axes.labelsize':     FONT_SIZE_MAX,
    'xtick.labelsize':    FONT_SIZE_MIN,
    'ytick.labelsize':    FONT_SIZE_MIN,
    'legend.fontsize':    FONT_SIZE_MIN,
    'figure.titlesize':   FONT_SIZE_MAX,
    'figure.dpi':         DPI,
    'savefig.dpi':        DPI,
    'savefig.bbox':       'tight',
    'savefig.pad_inches': 0.15,
    'figure.facecolor':   'white',
    'axes.facecolor':     'white',
    'savefig.facecolor':  'white',
    'font.family':        'sans-serif',
})
sns.set_style('whitegrid')

# Create output directories
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f'Results will be saved to : {os.path.abspath(RESULTS_DIR)}')
print(f'Model  will be saved to  : {os.path.abspath(MODEL_SAVE_PATH)}')

## 3. Load Data

Uses `SkinLesionDataLoader` from `data_loader.py` — same as `train.py` line 18.

In [ ]:
# 1. Load Data  (matches train.py → train() step 1)
print('Loading data...')
loader = SkinLesionDataLoader(
    data_dir=DATA_DIR,
    csv_path=CSV_PATH,
    batch_size=BATCH_SIZE
)
train_ds, val_ds = loader.load_data()

print(f'\nClasses ({loader.num_classes}): {loader.classes}')

# Quick peek at a batch
for imgs, labels in train_ds.take(1):
    print(f'Batch image shape : {imgs.shape}')
    print(f'Batch labels shape: {labels.shape}')

## 4. Build Model

Uses `build_efficientnetv2_model` + `compile_model` from `model.py` — same as `train.py` lines 23-24.

In [ ]:
# 2. Build Model  (matches train.py → train() step 2)
print('Building model...')
model = build_efficientnetv2_model(num_classes=loader.num_classes)
model = compile_model(model)
model.summary()

## 5. Compute Class Weights

Same logic as `train.py` lines 46-55.

In [ ]:
# Compute Class Weights to handle imbalance  (matches train.py → train() step 3)
print('Computing class weights...')
all_labels = []
for _, labels in train_ds:
    all_labels.extend(labels.numpy())

classes = np.unique(all_labels)
weights = compute_class_weight('balanced', classes=classes, y=all_labels)
class_weight_dict = {int(c): w for c, w in zip(classes, weights)}
print('Class Weights:', class_weight_dict)

## 6. Train Model

Same callbacks and `model.fit()` as `train.py` lines 29-65.

In [ ]:
# 3. Setup Callbacks  (matches train.py → train() step 3)
callbacks = [
    ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        save_best_only=True,
        monitor='val_loss',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    TensorBoard(log_dir=LOG_DIR)
]

# 4. Train Model  (matches train.py → train() step 4)
print('Starting training...\n')
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict
)

print(f'\nTraining complete. Best model saved to: {MODEL_SAVE_PATH}')

## 7. Save Training History Plot

Matches `train.py → save_training_history()` — 300 DPI, font sizes 10-12.

In [ ]:
# save_training_history()  — same logic as train.py
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax = axes[0]
ax.plot(history.history['accuracy'],    linewidth=2, marker='o', markersize=4, label='Train Accuracy')
ax.plot(history.history['val_accuracy'], linewidth=2, marker='s', markersize=4, label='Val Accuracy')
ax.set_title('Model Accuracy', fontsize=FONT_SIZE_MAX, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=FONT_SIZE_MAX)
ax.set_xlabel('Epoch', fontsize=FONT_SIZE_MAX)
ax.legend(loc='lower right', fontsize=FONT_SIZE_MIN)
ax.tick_params(axis='both', labelsize=FONT_SIZE_MIN)
ax.grid(True, alpha=0.3)

# Loss
ax = axes[1]
ax.plot(history.history['loss'],    linewidth=2, marker='o', markersize=4, label='Train Loss')
ax.plot(history.history['val_loss'], linewidth=2, marker='s', markersize=4, label='Val Loss')
ax.set_title('Model Loss', fontsize=FONT_SIZE_MAX, fontweight='bold')
ax.set_ylabel('Loss', fontsize=FONT_SIZE_MAX)
ax.set_xlabel('Epoch', fontsize=FONT_SIZE_MAX)
ax.legend(loc='upper right', fontsize=FONT_SIZE_MIN)
ax.tick_params(axis='both', labelsize=FONT_SIZE_MIN)
ax.grid(True, alpha=0.3)

fig.tight_layout()

history_path = os.path.join(RESULTS_DIR, 'training_history.png')
fig.savefig(history_path, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved training history plot to {history_path}')

## 8. Evaluate Model

Same evaluation loop as `train.py` lines under `if __name__`.

In [ ]:
# Evaluate on validation set  (matches train.py __main__ block)
print('Evaluating on validation set...')
y_true = []
y_pred = []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(f'Total validation samples: {len(y_true)}')

### 8a. Confusion Matrix

Matches `train.py → save_confusion_matrix()`.

In [ ]:
# save_confusion_matrix()  — same logic as train.py
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=loader.classes,
    yticklabels=loader.classes,
    linewidths=0.5,
    linecolor='gray',
    annot_kws={'size': FONT_SIZE_MIN, 'fontweight': 'bold'},
    cbar_kws={'shrink': 0.8},
    ax=ax
)
ax.set_title('Confusion Matrix', fontsize=FONT_SIZE_MAX, fontweight='bold', pad=15)
ax.set_ylabel('True Label', fontsize=FONT_SIZE_MAX)
ax.set_xlabel('Predicted Label', fontsize=FONT_SIZE_MAX)
ax.tick_params(axis='both', labelsize=FONT_SIZE_MIN)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.setp(ax.get_yticklabels(), rotation=0)

fig.tight_layout()

cm_path = os.path.join(RESULTS_DIR, 'confusion_matrix.png')
fig.savefig(cm_path, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved confusion matrix to {cm_path}')

### 8b. Classification Report → CSV

Matches `train.py → save_classification_report_csv()`.

In [ ]:
# save_classification_report_csv()  — same logic as train.py
report_dict = classification_report(
    y_true, y_pred,
    target_names=loader.classes,
    output_dict=True
)

# Per-class rows
rows = []
for cls_name in loader.classes:
    metrics = report_dict[cls_name]
    rows.append({
        'Class':     cls_name,
        'Precision': round(metrics['precision'], 4),
        'Recall':    round(metrics['recall'],    4),
        'F1-Score':  round(metrics['f1-score'],  4),
        'Support':   int(metrics['support'])
    })

# Blank separator row
rows.append({
    'Class': '', 'Precision': '', 'Recall': '', 'F1-Score': '', 'Support': ''
})

# Accuracy row
accuracy_val = report_dict['accuracy']
total_support = int(report_dict['weighted avg']['support'])
rows.append({
    'Class':     'accuracy',
    'Precision': '',
    'Recall':    '',
    'F1-Score':  round(accuracy_val, 4),
    'Support':   total_support
})

# Macro / weighted average rows
for avg_key in ['macro avg', 'weighted avg']:
    m = report_dict[avg_key]
    rows.append({
        'Class':     avg_key,
        'Precision': round(m['precision'], 4),
        'Recall':    round(m['recall'],    4),
        'F1-Score':  round(m['f1-score'],  4),
        'Support':   int(m['support'])
    })

report_df = pd.DataFrame(rows)

csv_path = os.path.join(RESULTS_DIR, 'classification_report.csv')
report_df.to_csv(csv_path, index=False)

print(f'Saved classification report CSV to {csv_path}\n')
print(report_df.to_string(index=False))

### 8c. Classification Report — Visual Table Image

Matches `train.py → save_classification_report_image()`.

In [ ]:
# save_classification_report_image()  — same logic as train.py
display_df = report_df.copy()

fig, ax = plt.subplots(figsize=(10, max(4, 0.5 * len(display_df) + 1.5)))
ax.axis('off')
ax.set_title('Classification Report', fontsize=FONT_SIZE_MAX, fontweight='bold', pad=20)

table = ax.table(
    cellText=display_df.values,
    colLabels=display_df.columns,
    cellLoc='center',
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(FONT_SIZE_MIN)
table.scale(1.2, 1.8)

# Header styling
for col_idx in range(len(display_df.columns)):
    cell = table[0, col_idx]
    cell.set_facecolor('#2c3e50')
    cell.set_text_props(color='white', fontweight='bold', fontsize=FONT_SIZE_MAX)
    cell.set_edgecolor('white')

# Row styling
num_class_rows = len(loader.classes)
for row_idx in range(1, len(display_df) + 1):
    for col_idx in range(len(display_df.columns)):
        cell = table[row_idx, col_idx]
        cell.set_edgecolor('#bdc3c7')

        if row_idx <= num_class_rows:
            cell.set_facecolor('#ecf0f1' if row_idx % 2 == 0 else '#ffffff')
        elif row_idx == num_class_rows + 1:
            cell.set_facecolor('#f8f9fa')
            cell.set_edgecolor('#f8f9fa')
        else:
            cell.set_facecolor('#d5e8d4')
            cell.set_text_props(fontweight='bold')

fig.tight_layout()

report_img_path = os.path.join(RESULTS_DIR, 'classification_report.png')
fig.savefig(report_img_path, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved classification report image to {report_img_path}')

## 9. Per-Class Performance Bar Chart

Matches `train.py → save_per_class_performance()`.

In [ ]:
# save_per_class_performance()  — same logic as train.py
class_metrics_df = report_df[report_df['Class'].isin(loader.classes)].copy()
class_metrics_df = class_metrics_df.set_index('Class')
metric_cols = ['Precision', 'Recall', 'F1-Score']
class_metrics_df[metric_cols] = class_metrics_df[metric_cols].astype(float)

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(loader.classes))
bar_width = 0.25
colors = ['#3498db', '#2ecc71', '#e74c3c']

for i, (metric, color) in enumerate(zip(metric_cols, colors)):
    bars = ax.bar(
        x + i * bar_width, class_metrics_df[metric], bar_width,
        label=metric, color=color, edgecolor='white', linewidth=0.5
    )
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2., height + 0.01,
                f'{height:.2f}', ha='center', va='bottom',
                fontsize=FONT_SIZE_MIN - 2, fontweight='bold'
            )

ax.set_title('Per-Class Performance Metrics', fontsize=FONT_SIZE_MAX, fontweight='bold')
ax.set_ylabel('Score', fontsize=FONT_SIZE_MAX)
ax.set_xlabel('Class', fontsize=FONT_SIZE_MAX)
ax.set_xticks(x + bar_width)
ax.set_xticklabels(loader.classes, rotation=45, ha='right', fontsize=FONT_SIZE_MIN)
ax.tick_params(axis='y', labelsize=FONT_SIZE_MIN)
ax.legend(fontsize=FONT_SIZE_MIN, loc='upper right')
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)

fig.tight_layout()

bar_path = os.path.join(RESULTS_DIR, 'per_class_performance.png')
fig.savefig(bar_path, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved per-class performance chart to {bar_path}')

## 10. Summary

In [ ]:
print('\n' + '='*60)
print('  📁  Files saved to:', os.path.abspath(RESULTS_DIR))
print('='*60)

for f in sorted(os.listdir(RESULTS_DIR)):
    fpath = os.path.join(RESULTS_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  📄 {f:<40s}  ({size_kb:>8.1f} KB)')

print(f'\n  🧠 Model saved to: {os.path.abspath(MODEL_SAVE_PATH)}')
print('='*60)
print(f'\nAll results saved to: {os.path.abspath(RESULTS_DIR)}')